In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
class State(TypedDict):
    count: int
# Define a simple node that increments a counter
def add_one(state: State):
    return {"count": state["count"] + 1}
# Build the graph
builder = StateGraph(State)
builder.add_node("add_one", add_one)
builder.add_edge(START, "add_one")
builder.add_edge("add_one", END)
# Initialize the checkpointer and compile the graph
checkpointer = InMemorySaver()

graph = builder.compile(checkpointer=checkpointer)

# Run the graph multiple times or let a multi-step graph run
result=graph.invoke({"count": 0}, config=config)
print(result)
# List all checkpoints stored in memory for this thread
# 1. Use the recommended high-level graph API to view all past checkpoints
config = {"configurable": {"thread_id": "my-thread-1"}}
history = list(graph.get_state_history(config))

# Let's safely print what actually exists in your thread
print("Available checkpoints for this thread:")
for state in history:
    print(f"- ID: {state.config['configurable']['checkpoint_id']} | Values: {state.values}")
# 2. Pick a valid checkpoint ID safely from the list
if history:
    # Let's pick the oldest or a specific one from the history array
    target_checkpoint_id = history[-1].config["configurable"]["checkpoint_id"]
    # 3. Retrieve it safely using graph.get_state instead of checkpointer.get_tuple
    specific_config = {
        "configurable": {
            "thread_id": "my-thread-1",
            "checkpoint_id": target_checkpoint_id
        }
    }
    specific_state = graph.get_state(specific_config)
        # Safely print the values
    print("\nSuccessfully retrieved state values:")
    print(specific_state.values)    
else:
    print("No history found for this thread ID. Ensure the graph has run first!")
for state in graph.get_state_history(config):
    cp_id = state.config["configurable"]["checkpoint_id"]
    values=state.values
    metadata=state.metadata
    next_node=state.next
    print(f"📍 Checkpoint ID : {cp_id}")
    print(f"🔢 State Values   : {values}")
    print(f"ℹ️  Metadata       : {metadata}")
    print(f"➡️  Next Node      : {next_node if next_node else 'None (Graph Finished/Ended)'}")
    print("-" * 50)


{'count': 1}
Available checkpoints for this thread:
- ID: 1f1b8b7a-cb63-69e5-8001-f0952540f12c | Values: {'count': 1}
- ID: 1f1b8b7a-cb5f-654f-8000-f28ef39db79b | Values: {'count': 0}
- ID: 1f1b8b7a-cb47-6feb-bfff-21c06854d29f | Values: {}

Successfully retrieved state values:
{}
📍 Checkpoint ID : 1f1b8b7a-cb63-69e5-8001-f0952540f12c
🔢 State Values   : {'count': 1}


NameError: name 'metadata' is not defined